# Introduction

This notebook presents an end-to-end **credit card fraud detection** workflow: **binary classification** on transactional data with strong **class imbalance**.

**Objectives**
- Build reproducible pipelines (`StandardScaler` → **SMOTE** → classifier) to avoid data leakage.
- Tune a **Random Forest** with **GridSearchCV** using **F1-score** (focus on the minority fraud class).
- Compare models using **cross-validation** on training data plus a final **hold-out test**; persist metrics, model, and figures.

**Target:** `Class` — `0` legitimate, `1` fraud.

**Environment:** Python 3.10+ recommended. Install dependencies with:

`pip install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn joblib` — optional: `xgboost` (otherwise histogram gradient boosting from sklearn is used).

## Configuration & artifact paths

Outputs are written to `ARTIFACT_DIR`. **GridSearchCV** (Random Forest) and **reporting cross-validation** use **stratified subsamples** of `X_train` for laptop-friendly runtimes; all **final pipelines** still `fit` on the **full training split** before test evaluation.

In [ ]:
from pathlib import Path

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

DATA_PATH = BASE_DIR / "creditcard.csv"
if not DATA_PATH.exists():
    DATA_PATH = Path("creditcard.csv")
ARTIFACT_DIR = BASE_DIR
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2

# RandomForest grid search: smaller stratified train slice keeps runtime reasonable; final RF refits on full train.
GRIDSEARCH_TRAIN_SAMPLES = 25_000
CV_FOLDS = 2

# Cross-validation (reporting): k-fold metrics on training data only — SMOTE runs inside the pipeline per fold (no test leakage).
CV_SCORING_FOLDS = 3
CV_SCORING_MAX_SAMPLES = 35_000  # stratified cap for CV scoring speed; does not change train/test split

FIG_KW = {"dpi": 150, "bbox_inches": "tight"}

print(f"Dataset path: {DATA_PATH.resolve()}")
print(f"Artifacts:      {ARTIFACT_DIR.resolve()}")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Place creditcard.csv in {BASE_DIR} or set DATA_PATH.")

# Data Understanding

We load the CSV, inspect schema and basic statistics, quantify missingness, and export EDA figures. Plots use consistent typography and axis labels suitable for reports or slides.

In [ ]:
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    auc,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    from xgboost import XGBClassifier

    HAS_XGB = True
except ImportError:
    HAS_XGB = False

sns.set_theme(style="whitegrid", context="notebook", palette="deep")
# Readable defaults for publication-style figures
plt.rcParams.update(
    {
        "figure.figsize": (10, 5.5),
        "figure.titlesize": 14,
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 10,
        "axes.grid": True,
        "grid.alpha": 0.3,
    }
)

# Consistent semantic colors: legitimate vs fraud
COLOR_LEGIT = "#2e7d32"
COLOR_FRAUD = "#c62828"
COLOR_NEUTRAL = "#1565c0"


def save_fig(path: Path, **kw) -> None:
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(p, **{**FIG_KW, **kw})
    plt.close()
    print(f"Saved: {p.resolve()}")


def missing_value_report(df: pd.DataFrame) -> pd.DataFrame:
    m = df.isnull().sum()
    pct = (m / len(df) * 100).round(4)
    return pd.DataFrame({"count": m, "pct": pct})


def plot_class_distribution(df: pd.DataFrame, target: str, out_path: Path) -> None:
    """Bar chart of Legitimate vs Fraud counts (full dataset before split)."""
    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    vc = df[target].value_counts().sort_index()
    xlabs = ["Legitimate (0)", "Fraud (1)"]
    cols = [COLOR_LEGIT, COLOR_FRAUD]
    ax.bar(xlabs[: len(vc)], vc.values, color=cols[: len(vc)], edgecolor="white", linewidth=1.1)
    ax.set_title("Class imbalance: transaction counts", fontweight="semibold")
    ax.set_xlabel("Label")
    ax.set_ylabel("Number of transactions")
    n = len(df)
    for i, (xi, vi) in enumerate(zip(range(len(vc)), vc.values)):
        ax.text(xi, vi + n * 0.002, f"{vi:,}\n({100 * vi / n:.2f}%)", ha="center", fontsize=9)
    plt.tight_layout()
    save_fig(out_path)


def plot_correlation_heatmap(df: pd.DataFrame, out_path: Path, exclude=None):
    """Pearson correlation of numeric inputs (target excluded to avoid trivial correlations)."""
    exclude = exclude or []
    num = df.select_dtypes(include=[np.number]).drop(columns=exclude, errors="ignore")
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(
        num.corr(),
        cmap="RdBu_r",
        center=0,
        ax=ax,
        square=False,
        cbar_kws={"shrink": 0.55, "label": "Pearson r"},
    )
    ax.set_title("Linear associations between numeric features", fontweight="semibold", pad=12)
    plt.tight_layout()
    save_fig(out_path)


def plot_feature_distributions(df: pd.DataFrame, cols: list, target: str, out_path: Path):
    """Histograms + boxplots by class for key interpretable / high-signal features."""
    cols = [c for c in cols if c in df.columns and c != target]
    n = len(cols)
    fig, axes = plt.subplots(n, 2, figsize=(11, 3.3 * max(n, 1)))
    if n == 1:
        axes = np.asarray([axes])
    for i, col in enumerate(cols):
        sns.histplot(df[col], bins=50, kde=True, ax=axes[i, 0], color=COLOR_NEUTRAL, alpha=0.85)
        axes[i, 0].set_title(f"Distribution — {col}", fontweight="medium")
        axes[i, 0].set_xlabel(col)
        axes[i, 0].set_ylabel("Count")
        sns.boxplot(
            data=df,
            x=target,
            y=col,
            ax=axes[i, 1],
            palette=[COLOR_LEGIT, COLOR_FRAUD],
            linewidth=1,
        )
        axes[i, 1].set_title(f"{col} by transaction type", fontweight="medium")
        axes[i, 1].set_xlabel("Class (0 = legitimate, 1 = fraud)")
        axes[i, 1].set_ylabel(col)
    fig.suptitle(
        "Exploratory feature distributions",
        fontsize=14,
        fontweight="semibold",
        y=1.01,
    )
    plt.tight_layout()
    save_fig(out_path)


def make_fraud_pipeline(classifier) -> ImbPipeline:
    """Reusable imblearn pipeline: scale numerics → SMOTE (fit only) → estimator."""
    return ImbPipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("smote", SMOTE(random_state=RANDOM_STATE)),
            ("clf", classifier),
        ]
    )


def stratified_train_subset(
    X: pd.DataFrame, y: pd.Series, n_samples: int, random_state: int
):
    """Draw a smaller stratified sample (e.g. for grid search or CV scoring) preserving fraud rate."""
    n_samples = min(n_samples, len(X))
    if n_samples == len(X):
        return X, y
    X_sub, _, y_sub, _ = train_test_split(
        X,
        y,
        train_size=n_samples,
        stratify=y,
        random_state=random_state,
    )
    return X_sub, y_sub


def clf_params_from_grid(best_params: dict) -> dict:
    """Map GridSearch keys `clf__*` to constructor kwargs for RandomForestClassifier."""
    return {k.split("__", 1)[1]: v for k, v in best_params.items() if k.startswith("clf__")}


def evaluate_pipeline(pipe: ImbPipeline, X_test, y_test, name: str) -> dict:
    """Out-of-sample metrics on the raw test set (pipelines handle scaling; no SMOTE on X_test)."""
    y_pred = pipe.predict(X_test)
    proba = None
    clf = pipe.named_steps["clf"]
    if hasattr(clf, "predict_proba"):
        proba = pipe.predict_proba(X_test)[:, 1]
    elif hasattr(clf, "decision_function"):
        proba = pipe.decision_function(X_test)
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": np.nan if proba is None else roc_auc_score(y_test, proba),
    }


def cross_val_scores_for_model(
    fitted_pipe: ImbPipeline,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    model_name: str,
    cv_splitter,
    max_samples: int,
    random_state: int,
) -> dict:
    """
    Stratified k-fold scores on training data only (clone unfitted pipeline per run).
    Uses a row cap for speed; final models still train on full X_train elsewhere.
    """
    X_cv, y_cv = stratified_train_subset(X_train, y_train, max_samples, random_state)
    sk = clone(fitted_pipe)
    scores = cross_validate(
        sk,
        X_cv,
        y_cv,
        cv=cv_splitter,
        scoring={"f1": "f1", "roc_auc": "roc_auc"},
        n_jobs=-1,
    )
    return {
        "Model": model_name,
        "CV_F1_mean": float(np.mean(scores["test_f1"])),
        "CV_F1_std": float(np.std(scores["test_f1"])),
        "CV_ROC_AUC_mean": float(np.mean(scores["test_roc_auc"])),
        "CV_ROC_AUC_std": float(np.std(scores["test_roc_auc"])),
    }


def plot_smote_effect(y_before, y_after, out_path: Path):
    """Illustrate minority oversampling on the training split only (demo / teaching)."""
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.5))
    xtick_labs = ["Legitimate (0)", "Fraud (1)"]
    for ax, yseries, title in zip(
        axes,
        (y_before, y_after),
        (
            "Before SMOTE (original train counts)",
            "After SMOTE (balanced train — fit-time only)",
        ),
    ):
        s = pd.Series(yseries).value_counts().sort_index()
        ax.bar(xtick_labs[: len(s)], s.values, color=[COLOR_LEGIT, COLOR_FRAUD][: len(s)], edgecolor="white")
        ax.set_title(title, fontweight="medium")
        ax.set_xlabel("Class")
        ax.set_ylabel("Number of samples")
    fig.suptitle("SMOTE effect on training labels (no test data)", fontweight="semibold", y=1.02)
    plt.tight_layout()
    save_fig(out_path)

In [ ]:
# ---------------------------------------------------------------------------
# 1) Load raw CSV — path comes from the configuration cell
# ---------------------------------------------------------------------------
df = pd.read_csv(DATA_PATH)
TARGET = "Class"

# Coerce labels to int and drop rows where Class could not be parsed
if TARGET in df.columns:
    df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
    df = df.dropna(subset=[TARGET]).reset_index(drop=True)
    df[TARGET] = df[TARGET].astype(int)

print(f"Shape: {df.shape[0]:,} × {df.shape[1]}")
display(df.head())
print()
df.info()
print()
display(df.describe())

In [ ]:
# ---------------------------------------------------------------------------
# 2) Data quality + EDA charts (saved to ARTIFACT_DIR)
# ---------------------------------------------------------------------------
miss = missing_value_report(df)
print("Missing values:")
display(miss[miss["count"] > 0] if miss["count"].any() else miss.head())
print("Total missing cells:", int(miss["count"].sum()))

plot_class_distribution(df, TARGET, ARTIFACT_DIR / "class_distribution.png")
plot_correlation_heatmap(df, ARTIFACT_DIR / "correlation_heatmap.png", exclude=[TARGET])

dist_cols = ["Time", "Amount", "V12", "V14", "V17"]
plot_feature_distributions(df, dist_cols, TARGET, ARTIFACT_DIR / "feature_distributions.png")

# Data Preparation

- Impute or drop missing values if present; keep feature matrix and target aligned.
- **No categoricals** in the standard `creditcard.csv`; one-hot encoding is applied only when string columns exist.
- **Train/test split (80/20), stratified** on fraud rate.
- **SMOTE** is illustrated on **training data only** (after scaling), matching the pipeline order; the modeling section uses the same logic inside `imblearn.pipeline.Pipeline`.

In [ ]:
# ---------------------------------------------------------------------------
# 3) Feature matrix + target, simple imputation, optional one-hot encoding
# ---------------------------------------------------------------------------
feature_cols = [c for c in df.columns if c != TARGET]
X = df[feature_cols].copy()
y = df[TARGET].copy()

# Numeric-only median fill (creditcard.csv is usually complete; keeps pipeline robust)
if X.isnull().any().any():
    X = X.fillna(X.median(numeric_only=True))

cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
if cat_cols:
    X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
    print("Encoded:", cat_cols)
else:
    print("No categorical columns to encode.")

FEATURE_NAMES = list(X.columns)

# Stratified 80/20: preserves rare fraud proportion in train and test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print(
    f"Fraud rate train: {y_train.mean():.5f} | test: {y_test.mean():.5f}"
)

# Demo: same order as modeling pipeline (scale train → SMOTE on train). Test set is never resampled.
_sc = StandardScaler().fit(X_train)
X_train_s = _sc.transform(X_train)
_sm = SMOTE(random_state=RANDOM_STATE)
_, y_smote_demo = _sm.fit_resample(X_train_s, y_train)
plot_smote_effect(y_train, y_smote_demo, ARTIFACT_DIR / "smote_class_distribution.png")

# Modeling

Each estimator is wrapped in **`imblearn.pipeline.Pipeline`**: `StandardScaler` → `SMOTE` → classifier. Fitting applies scaling and SMOTE **only on training folds** inside `GridSearchCV`, and on full training data for final models.

**Random Forest:** `GridSearchCV` with stratified k-fold CV (`CV_FOLDS` in the config cell), **`scoring="f1"`**, on a stratified subset of `X_train` (for feasible runtime on laptops). Best hyperparameters are then used to refit a full pipeline on **all** of `X_train`.

In [ ]:
# ---------------------------------------------------------------------------
# Cross-validator for RandomForest grid search (few folds = faster)
# ---------------------------------------------------------------------------
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# Smaller stratified slice: hyperparameter search only; final RF uses all of X_train below
X_tune, y_tune = stratified_train_subset(
    X_train, y_train, GRIDSEARCH_TRAIN_SAMPLES, RANDOM_STATE
)
print(
    f"GridSearch RF on {len(X_tune):,} stratified train rows (full train {len(X_train):,} for refit)."
)

# --- Random Forest: GridSearchCV optimizes F1; pipeline applies SMOTE inside each CV fold ---
rf_pipe_search = make_fraud_pipeline(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
)
param_grid_rf = {
    "clf__n_estimators": [100, 200],
    "clf__max_depth": [25, None],
}

grid_rf = GridSearchCV(
    estimator=rf_pipe_search,
    param_grid=param_grid_rf,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    refit=True,
    verbose=1,
)
grid_rf.fit(X_tune, y_tune)
print("Best RF params:", grid_rf.best_params_)
print("Best inner-CV F1 (grid search):", round(grid_rf.best_score_, 4))

# Rebuild RF with tuned settings and fit on **full** training set
rf_best_kwargs = clf_params_from_grid(grid_rf.best_params_)
pipe_rf = make_fraud_pipeline(
    RandomForestClassifier(**rf_best_kwargs, random_state=RANDOM_STATE, n_jobs=-1)
)
pipe_rf.fit(X_train, y_train)

# --- Baseline models: same pipeline shape (scale → SMOTE → clf) ---
pipe_lr = make_fraud_pipeline(
    LogisticRegression(max_iter=5000, solver="lbfgs", random_state=RANDOM_STATE)
)
pipe_lr.fit(X_train, y_train)

pipe_dt = make_fraud_pipeline(
    DecisionTreeClassifier(max_depth=10, random_state=RANDOM_STATE)
)
pipe_dt.fit(X_train, y_train)

# Gradient boosting: XGBoost if installed, else fast sklearn histogram boosting
if HAS_XGB:
    gb_est = XGBClassifier(
        n_estimators=150,
        max_depth=6,
        learning_rate=0.08,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        verbosity=0,
        n_jobs=-1,
    )
    gb_label = "Gradient Boosting (XGBoost)"
else:
    gb_est = HistGradientBoostingClassifier(
        max_iter=150,
        max_depth=7,
        learning_rate=0.08,
        random_state=RANDOM_STATE,
    )
    gb_label = "Gradient Boosting (HistGradientBoosting)"

pipe_gb = make_fraud_pipeline(gb_est)
pipe_gb.fit(X_train, y_train)

# Dictionary of **fitted** pipelines — used for CV summary, test evaluation, and saving the best model
fitted_pipelines = {
    "Logistic Regression": pipe_lr,
    "Decision Tree": pipe_dt,
    "Random Forest (tuned)": pipe_rf,
    gb_label: pipe_gb,
}
print("Fitted models:", list(fitted_pipelines.keys()))

# Evaluation

**Cross-validation (train only):** mean ± std **F1** and **ROC-AUC** over stratified folds on a capped subset of `X_train` (each fold: scale → SMOTE → fit on fold-train only). This summarises stability before the **held-out test** step.

**Test metrics:** on the **test set** at natural imbalance (no SMOTE on test). `predict` uses the scaler learned on full train; resampling is training-time only.

In [ ]:
# ---------------------------------------------------------------------------
# A) Cross-validation on training data (clone + refit per fold; SMOTE inside pipeline)
# ---------------------------------------------------------------------------
cv_splitter = StratifiedKFold(
    n_splits=CV_SCORING_FOLDS, shuffle=True, random_state=RANDOM_STATE
)
print(
    f"CV scoring: {CV_SCORING_FOLDS}-fold, up to {CV_SCORING_MAX_SAMPLES:,} stratified train rows per model."
)
cv_rows = [
    cross_val_scores_for_model(
        pipe,
        X_train,
        y_train,
        name,
        cv_splitter,
        CV_SCORING_MAX_SAMPLES,
        RANDOM_STATE,
    )
    for name, pipe in fitted_pipelines.items()
]
cv_df = pd.DataFrame(cv_rows)
print("\nCross-validation metrics (train-only subset; mean / std across folds):")
display(cv_df.round(4))

# ---------------------------------------------------------------------------
# B) Held-out test performance (never seen during fit / SMOTE)
# ---------------------------------------------------------------------------
rows = [
    evaluate_pipeline(pipe, X_test, y_test, name)
    for name, pipe in fitted_pipelines.items()
]
results_df = pd.DataFrame(rows).merge(cv_df, on="Model", how="left")

# Readable column order for CSV / reports
col_order = [
    "Model",
    "CV_F1_mean",
    "CV_F1_std",
    "CV_ROC_AUC_mean",
    "CV_ROC_AUC_std",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC",
]
results_df = results_df[[c for c in col_order if c in results_df.columns]]

out_csv = ARTIFACT_DIR / "model_results.csv"
results_df.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv.resolve()}")

print("\nCombined CV + test results:")
display(results_df.round(4))

sorted_f1 = results_df.sort_values(["F1", "ROC-AUC"], ascending=False).reset_index(drop=True)
sorted_auc = results_df.sort_values(["ROC-AUC", "F1"], ascending=False).reset_index(drop=True)
print("\nSorted by test F1, then test ROC-AUC:")
display(sorted_f1.round(4))
print("\nSorted by test ROC-AUC, then test F1:")
display(sorted_auc.round(4))

# Results

The **best model** is chosen from **test F1** with tie-break **test ROC-AUC** (not from CV alone, so the test set remains an unbiased final check). The **full fitted pipeline** is saved as **`best_model.pkl`** for reproducibility. Note: SMOTE is a **training-time** resampler; production scoring paths often rely on the learned **scaler + classifier** and separately tuned **decision thresholds**, not on generating synthetic points at inference.

In [ ]:
# Pick winner by test F1 (tie-break ROC-AUC on test)
best_name = sorted_f1.iloc[0]["Model"]
best_pipe = fitted_pipelines[best_name]

print("=" * 64)
print("BEST MODEL (by test F1): ", best_name)
print("=" * 64)
display(sorted_f1.head(1).round(4))

joblib.dump(best_pipe, ARTIFACT_DIR / "best_model.pkl")
print(f"Saved: {(ARTIFACT_DIR / 'best_model.pkl').resolve()}")

y_pred_best = best_pipe.predict(X_test)
y_proba_best = best_pipe.predict_proba(X_test)[:, 1]
_label_order = ["Legitimate", "Fraud"]

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_best,
    ax=ax,
    display_labels=_label_order,
    cmap="Blues",
    colorbar=True,
)
ax.set_title(f"Confusion matrix (test set)\n{best_name}", fontweight="semibold")
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
plt.tight_layout()
save_fig(ARTIFACT_DIR / "confusion_matrix.png")

fpr, tpr, _ = roc_curve(y_test, y_proba_best)
roc_auc_val = auc(fpr, tpr)
fig, ax = plt.subplots(figsize=(7.5, 5.5))
ax.plot(
    fpr,
    tpr,
    lw=2.5,
    color=COLOR_NEUTRAL,
    label=f"Model (AUC = {roc_auc_val:.4f})",
)
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", lw=1.2, label="Random classifier (AUC = 0.5)")
ax.fill_between(fpr, tpr, alpha=0.12, color=COLOR_NEUTRAL)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.set_xlabel("False positive rate (1 − specificity)")
ax.set_ylabel("True positive rate (recall on fraud)")
ax.set_title(f"Receiver operating characteristic (ROC)\n{best_name}", fontweight="semibold")
ax.legend(loc="lower right", frameon=True)
ax.grid(True, alpha=0.35)
plt.tight_layout()
save_fig(ARTIFACT_DIR / "roc_curve.png")

clf = best_pipe.named_steps["clf"]
fig, ax = plt.subplots(figsize=(10, 8))
if hasattr(clf, "feature_importances_"):
    imp = clf.feature_importances_
    xlab = "Feature importance (tree-based model)"
elif hasattr(clf, "coef_"):
    imp = np.abs(clf.coef_).ravel()
    xlab = "Absolute logistic coefficient (linear model)"
else:
    imp = None

if imp is not None:
    names = np.array(FEATURE_NAMES)
    k = min(25, len(imp))
    order = np.argsort(imp)[::-1][:k]
    sns.barplot(x=imp[order], y=names[order], ax=ax, orient="h", color=COLOR_NEUTRAL)
    ax.set_xlabel(xlab)
    ax.axvline(0, color="gray", lw=0.8, alpha=0.6)
    ax.set_title(
        f"Top {k} features ranked by model signal\n{best_name}",
        fontweight="semibold",
    )
else:
    ax.text(0.5, 0.5, "No feature_importances_ or coef_ available.", ha="center")
    ax.set_title(f"Feature importance — {best_name}", fontweight="semibold")
plt.tight_layout()
save_fig(ARTIFACT_DIR / "feature_importance.png")

print("\nFigures and model written to:", ARTIFACT_DIR.resolve())

## Business interpretation

### Why class imbalance matters

Legitimate transactions vastly outnumber fraud. A naive model that **always predicts “legitimate”** can still show **high accuracy** while failing completely at fraud detection. Business decisions must rely on metrics that reflect **minority-class performance**, not accuracy alone.

### Why F1-score is central here

**F1** is the harmonic mean of **precision** and **recall** on the fraud class. It rewards a **balance** between catching fraud (**recall**) and avoiding too many false alarms (**precision**). Random Forest hyperparameters are chosen with **GridSearchCV** and **`scoring="f1"`** on inner folds; the notebook also reports **outer-style CV means** (F1 and ROC-AUC) for every model so you can compare stability before looking at the single test split.

### Cost of false negatives (undetected fraud)

A **false negative** is a fraudulent transaction classified as legitimate. Consequences include **direct financial loss**, **customer trust** damage, regulatory scrutiny, and operational cost of investigations. In many settings **recall** on fraud is therefore prioritized, sometimes at the expense of precision — the right trade-off depends on **costs** (false negative vs false positive) defined with stakeholders.

### Why the selected model is preferred

The **best model** in this notebook is the one with the highest **test F1** (with **ROC-AUC** as tie-breaker). **ROC-AUC** summarizes how well the model **ranks** fraud vs non-fraud across thresholds; a higher value usually indicates better separability. The chosen model offers the strongest combined ranking and F1 on held-out data among the candidates, under the same preprocessing and SMOTE-in-pipeline training protocol. **Feature importance** (or coefficient magnitudes for linear models) highlights which inputs drive predictions and can guide monitoring and domain review — remembering that anonymized `V*` features are PCA-like and not as directly actionable as raw merchant attributes.

# Conclusion

This notebook loads `creditcard.csv`, explores imbalance and correlations, applies stratified **80/20** splitting, and fits **imblearn pipelines** (`StandardScaler` → **SMOTE** → classifier) so **SMOTE never touches the test set**. **Random Forest** is tuned with **GridSearchCV** on **F1**. Each model is summarized with **stratified k-fold cross-validation** (F1 and ROC-AUC on a capped training subset) and with **held-out test metrics** (Accuracy, Precision, Recall, F1, ROC-AUC). **`model_results.csv`** stores both CV and test columns; **`best_model.pkl`** stores the winning pipeline; figures include **`class_distribution.png`**, **`correlation_heatmap.png`**, **`smote_class_distribution.png`**, **`feature_distributions.png`**, **`confusion_matrix.png`**, **`roc_curve.png`**, and **`feature_importance.png`**.

For a production system, next steps typically include **calibration**, **threshold optimization** using real **costs**, **temporal validation** (fraud drift), and **fairness** review across customer segments.